# 02 — Local BF16 baseline for either model profile

This notebook evaluates the exact pinned local BF16 checkpoint that a later LoRA
notebook will tune. The default is the one-GPU Nano workshop profile. Change one
variable to create the separate Lightning advanced baseline.

**Required runtime:** use the NeMo container Jupyter opened by
`launchable/setup.sh` through the Secure Link on host port **8889**.


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from nemotron_ft_lab.model_profiles import get_model_profile

MODEL_PROFILE_NAME = os.environ.get('NEMOTRON_MODEL_PROFILE', 'nano9b_workshop')
# To prepare Notebook 04 instead, set: MODEL_PROFILE_NAME = 'lightning35_advanced'
PROFILE = get_model_profile(MODEL_PROFILE_NAME)
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
EVALUATION_DIR = ARTIFACTS_DIR / 'evaluation' / PROFILE.artifact_slug
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PATH = EVALUATION_DIR / 'baseline_local_bf16_text2sql.json'
print(json.dumps(PROFILE.as_dict(), indent=2))
print('Python:', sys.executable)
print('Baseline report:', BASELINE_PATH)


## 1. Verify the container, GPU, model identity, and frozen holdout


In [ ]:
subprocess.run([
    sys.executable, 'scripts/preflight.py', '--profile', 'inference',
    '--model-profile', PROFILE.name,
], check=True)
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py',
    '--output-dir', str(DATA_DIR), '--evaluation-only',
], check=True)

import torch

from nemotron_ft_lab.data import read_jsonl

eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
manifest = json.loads((DATA_DIR / 'evaluation_manifest.json').read_text())
print('Model:', PROFILE.model_id)
print('Pinned revision:', PROFILE.revision)
print('System prompt:', repr(PROFILE.system_prompt))
print('GPU(s):', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Evaluation rows:', len(eval_rows), manifest['difficulty_distribution'])


## 2. Run local vLLM in an isolated process

Process exit releases GPU memory before training. Raw predictions are saved
before SQL scoring and reused only when model, profile, prompt, Mamba cache,
generation settings, and evaluation hash all match. Nano automatically uses the
model-card-required float32 Mamba SSM cache and `/no_think` prompt.


In [ ]:
INFERENCE_GPUS = int(os.environ.get('NEMOTRON_INFERENCE_GPUS', '1'))
if not 1 <= INFERENCE_GPUS <= torch.cuda.device_count():
    raise RuntimeError(f'NEMOTRON_INFERENCE_GPUS must be between 1 and {torch.cuda.device_count()}.')
command = [
    sys.executable, 'scripts/evaluate_vllm.py',
    '--model-profile', PROFILE.name,
    '--model', PROFILE.model_id, '--revision', PROFILE.revision,
    '--data-dir', str(DATA_DIR), '--output', str(BASELINE_PATH),
    '--run-type', f'untuned-local-bf16-text2sql-{PROFILE.name}',
    '--tensor-parallel-size', str(INFERENCE_GPUS),
]
print('Launching:', ' '.join(command))
started = time.perf_counter()
subprocess.run(command, check=True)
print(f'Baseline process finished in {(time.perf_counter() - started) / 60:.1f} min')


In [ ]:
baseline = json.loads(BASELINE_PATH.read_text())
metrics = ('n', 'execution_accuracy', 'sql_valid_rate', 'sql_executable_rate', 'normalized_exact_match')
print(json.dumps({key: baseline[key] for key in metrics}, indent=2))
for row in baseline['rows'][:5]:
    print('\nQ:', row['question'])
    print('Gold:', row['expected_sql'])
    print('Generated:', row['generated'])
    print('Execution correct:', row['execution_correct'])


## Result contract

Notebook 03 or 04 must reuse this profile-namespaced report and the exact IDs in
it. Switching profiles here writes a different report; it never overwrites or
masquerades as the other model's baseline.
